In [1]:
import os
os.environ["PYSPARK_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import col, count, spark_partition_id, lit, rand, when

spark = SparkSession.builder \
    .appName("Week3_Day2") \
    .master("local[*]") \
    .getOrCreate()

# Create a moderately sized DataFrame (10,000 rows)
df_large = (
    spark.range(10000)
    .withColumn("department", 
        when(col("id") % 5 == 0, "Engineering")
        .when(col("id") % 5 == 1, "Data")
        .when(col("id") % 5 == 2, "HR")
        .when(col("id") % 5 == 3, "Finance")
        .otherwise("Marketing")
    )
    .withColumn("salary", (rand() * 50000 + 50000).cast("int"))
    .withColumn("region",
        when(col("id") % 3 == 0, "East Africa")
        .when(col("id") % 3 == 1, "West Africa")
        .otherwise("Southern Africa")
    )
)

In [3]:
df_large.show(5)
print(f"Total rows: {df_large.count()}")

+---+-----------+------+---------------+
| id| department|salary|         region|
+---+-----------+------+---------------+
|  0|Engineering| 81936|    East Africa|
|  1|       Data| 62612|    West Africa|
|  2|         HR| 64118|Southern Africa|
|  3|    Finance| 54546|    East Africa|
|  4|  Marketing| 65300|    West Africa|
+---+-----------+------+---------------+
only showing top 5 rows

Total rows: 10000


In [4]:
print(f"Number of partitions: {df_large.rdd.getNumPartitions()}")

Number of partitions: 4


In [5]:
df_large.withColumn("partition_id", spark_partition_id())\
        .groupBy("partition_id")\
        .count()\
        .orderBy("partition_id")\
        .show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 2500|
|           1| 2500|
|           2| 2500|
|           3| 2500|
+------------+-----+



In [6]:
df_2parts = df_large.repartition(2)

print(f"Before:{df_large.rdd.getNumPartitions()} partitions")
print(f"After:{df_2parts.rdd.getNumPartitions()} partitions")

df_2parts.withColumn("partition_id", spark_partition_id()) \
    .groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

Before:4 partitions
After:2 partitions
+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 5000|
|           1| 5000|
+------------+-----+



In [7]:
# Repartition by department
df_by_dept = df_large.repartition("department")

print(f"Partitions: {df_by_dept.rdd.getNumPartitions()}")

# What's in each partition?
df_by_dept.withColumn("partition_id", spark_partition_id()) \
    .groupBy("partition_id", "department") \
    .count() \
    .orderBy("partition_id", "department") \
    .show(20)

Partitions: 1
+------------+-----------+-----+
|partition_id| department|count|
+------------+-----------+-----+
|           0|       Data| 2000|
|           0|Engineering| 2000|
|           0|    Finance| 2000|
|           0|         HR| 2000|
|           0|  Marketing| 2000|
+------------+-----------+-----+



In [8]:
# Reduce from 8 partitions to 2 — NO shuffle
df_2parts = df_large.coalesce(2)

print(f"Before: {df_large.rdd.getNumPartitions()} partitions")
print(f"After:  {df_2parts.rdd.getNumPartitions()} partitions")

df_2parts.withColumn("partition_id", spark_partition_id()) \
    .groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

Before: 4 partitions
After:  2 partitions
+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 5000|
|           1| 5000|
+------------+-----+

